In [2]:
import numpy as np


def multi_head_attention(input_embeddings, query_projection_weights,
                         key_projection_weights, value_projection_weights,
                         output_projection_weights, number_of_heads):
    """
    Return output (seq_len, d_model) and weights
    (num_heads, seq_len, seq_len).
    """

    query = np.matmul(input_embeddings, query_projection_weights)
    key = np.matmul(input_embeddings, key_projection_weights)
    value = np.matmul(input_embeddings, value_projection_weights)

    sequence_length, model_dimension = input_embeddings.shape
    head_dimension = model_dimension // number_of_heads

    assert model_dimension % number_of_heads == 0, "d_model must be divisible by num_heads"

    query = query.reshape(sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)
    key = key.reshape(sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)
    value = value.reshape(sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)

    attention_scores = np.matmul(query, key.transpose(0, 2, 1)) / np.sqrt(head_dimension)

    attention_scores = attention_scores - np.max(attention_scores, axis=-1, keepdims=True)
    exponentiated_scores = np.exp(attention_scores)

    attention_weights = exponentiated_scores / np.sum(
        exponentiated_scores, axis=-1, keepdims=True
    )

    per_head_outputs = np.matmul(attention_weights, value)

    concatenated_heads = per_head_outputs.transpose(1, 0, 2).reshape(
        sequence_length, model_dimension
    )

    attention_output = np.matmul(concatenated_heads, output_projection_weights)

    return attention_output, attention_weights

sequence_length = 5
model_dimension = 8
number_of_heads = 2

sample_input_embeddings = np.random.rand(sequence_length, model_dimension)

sample_weight_query = np.random.rand(model_dimension, model_dimension)
sample_weight_key = np.random.rand(model_dimension, model_dimension)
sample_weight_value = np.random.rand(model_dimension, model_dimension)
sample_weight_output = np.random.rand(model_dimension, model_dimension)

multi_head_attention_output, multi_head_attention_weights = multi_head_attention(
    sample_input_embeddings,
    sample_weight_query,
    sample_weight_key,
    sample_weight_value,
    sample_weight_output,
    number_of_heads
)

print("Attention Weights:\n", multi_head_attention_weights)
print("\nAttention Output:\n", multi_head_attention_output)

print("\nOutput Shape:", multi_head_attention_output.shape)
print("Attention Weights Shape:", multi_head_attention_weights.shape)

Attention Weights:
 [[[0.19765987 0.21364109 0.14817247 0.29426117 0.1462654 ]
  [0.17042536 0.21934408 0.13944249 0.31711699 0.15367108]
  [0.17629013 0.21543089 0.14043238 0.31935195 0.14849465]
  [0.16669976 0.22032445 0.13490739 0.31813294 0.15993546]
  [0.16369989 0.21986295 0.14158066 0.30659683 0.16825967]]

 [[0.10979645 0.23606924 0.27952153 0.25436198 0.12025079]
  [0.10636083 0.2461742  0.28711568 0.2497961  0.1105532 ]
  [0.12315274 0.24118296 0.29090786 0.22729716 0.11745928]
  [0.09858525 0.24388027 0.29776956 0.25738309 0.10238183]
  [0.11040927 0.25056184 0.26756935 0.25407197 0.11738757]]]

Attention Output:
 [[7.50980091 8.77702466 8.22538418 5.97800385 8.55027667 8.5676891
  7.34872859 5.16048224]
 [7.52155123 8.80035692 8.23648416 5.98339148 8.56149874 8.57757707
  7.3640128  5.17902695]
 [7.51875431 8.79256683 8.23300566 5.97822476 8.55769767 8.57514993
  7.35925901 5.17489166]
 [7.51911554 8.80254504 8.23446516 5.98265518 8.55973019 8.57484652
  7.3634398  5.18211